# 机器学习算法实践 03：朴素贝叶斯文本分类——垃圾邮件识别

## 实验目的

本实验以 SMS Spam Collection 垃圾短信数据集为例，学习并实现朴素贝叶斯文本分类算法。

通过本实验理解贝叶斯分类、条件独立假设、先验概率、条件概率以及拉普拉斯平滑等基本概念，并掌握文本数据从原始文本到数值特征的转换过程。

实验首先使用词袋模型将短信文本转换为词频特征，然后从零实现多项式朴素贝叶斯分类算法，并研究拉普拉斯平滑对零概率问题的影响。最后使用 scikit-learn 中的 MultinomialNB 进行对比验证。

## 1. 朴素贝叶斯算法原理

朴素贝叶斯（Naive Bayes）是一类基于贝叶斯定理的概率分类算法。

对于待分类样本 $X$ 和类别 $C$，贝叶斯定理描述了在观察到特征 $X$ 后，样本属于类别 $C$ 的概率。

其中：

- $P(C)$：类别 $C$ 的先验概率；
- $P(X|C)$：类别为 $C$ 时观察到特征 $X$ 的条件概率；
- $P(X)$：观察到特征 $X$ 的概率；
- $P(C|X)$：观察到特征 $X$ 后属于类别 $C$ 的后验概率。

分类时，对于同一个样本 $X$，$P(X)$ 对所有类别都相同，因此只需要比较：

$$
P(X|C)P(C)
$$

选择该值最大的类别作为最终预测结果。

### 1.1 条件独立假设

朴素贝叶斯中的“朴素”来自一个重要假设：

> 在给定类别的条件下，各个特征之间相互独立。

对于文本分类任务，假设一封短信包含多个单词：

$$
X=(x_1,x_2,\ldots,x_n)
$$

朴素贝叶斯将联合条件概率近似为：

$$
P(X|C)
=
P(x_1|C)
P(x_2|C)
\cdots
P(x_n|C)
$$

因此，可以分别计算每个单词在某个类别下出现的概率，再将这些概率组合起来进行分类。

虽然现实文本中的单词并不完全独立，但这一简化假设能够显著降低概率计算的复杂度，并且在许多文本分类任务中仍能取得较好的效果。

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

print("环境准备完成！")

环境准备完成！


## 2. SMS Spam Collection 数据集

本实验使用 SMS Spam Collection 数据集进行垃圾短信分类。

数据集中每条短信包含一个类别标签和对应的短信文本：

- `ham`：正常短信；
- `spam`：垃圾短信。

机器学习任务是根据短信的文本内容，判断一条新的短信属于正常短信还是垃圾短信。

In [3]:
df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
print("数据集形状：", df.shape)

print("\n各类别样本数量：")
print(df["label"].value_counts())

print("\n缺失值数量：")
print(df.isnull().sum())

print("\n重复样本数量：")
print(df.duplicated().sum())

数据集形状： (5572, 2)

各类别样本数量：
label
ham     4825
spam     747
Name: count, dtype: int64

缺失值数量：
label      0
message    0
dtype: int64

重复样本数量：
403


In [5]:
label_counts = df["label"].value_counts()

print(label_counts)

print("\n类别比例：")
print(label_counts / len(df))

label
ham     4825
spam     747
Name: count, dtype: int64

类别比例：
label
ham     0.865937
spam    0.134063
Name: count, dtype: float64


In [6]:
X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("训练集大小：", len(X_train))
print("测试集大小：", len(X_test))

print("\n训练集类别分布：")
print(y_train.value_counts())

print("\n测试集类别分布：")
print(y_test.value_counts())

训练集大小： 4457
测试集大小： 1115

训练集类别分布：
label
ham     3859
spam     598
Name: count, dtype: int64

测试集类别分布：
label
ham     966
spam    149
Name: count, dtype: int64


## 3. 文本特征提取

机器学习模型不能直接对原始字符串进行数学计算，因此需要首先将短信文本转换为数值特征。

本实验使用词袋模型（Bag of Words）表示文本。

词袋模型首先建立词汇表，然后统计每条短信中各个单词出现的次数，从而将文本转换为一个数值向量。

例如，假设词汇表为：

`["free", "win", "meeting"]`

短信：

`"free free win"`

可以表示为：

$$
[2,1,0]
$$

表示：

- `free` 出现 2 次；
- `win` 出现 1 次；
- `meeting` 出现 0 次。

这种词频特征非常适合使用多项式朴素贝叶斯（Multinomial Naive Bayes）进行分类。

In [8]:
vectorizer = CountVectorizer()

X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

print("训练集词频矩阵形状：", X_train_counts.shape)
print("测试集词频矩阵形状：", X_test_counts.shape)

print("词汇表大小：", len(vectorizer.vocabulary_))

训练集词频矩阵形状： (4457, 7668)
测试集词频矩阵形状： (1115, 7668)
词汇表大小： 7668


In [9]:
sample_texts = [
    "free prize win",
    "free free meeting"
]

sample_vectorizer = CountVectorizer()

sample_matrix = sample_vectorizer.fit_transform(sample_texts)

print("词汇表：")
print(sample_vectorizer.get_feature_names_out())

print("\n词频矩阵：")
print(sample_matrix.toarray())

词汇表：
['free' 'meeting' 'prize' 'win']

词频矩阵：
[[1 0 1 1]
 [2 1 0 0]]


## 4. 从零实现多项式朴素贝叶斯

在完成文本的词频特征提取后，本节从零实现多项式朴素贝叶斯分类器。

对于类别 $C$ 和文本特征 $X$，分类时需要比较：

$$
P(C)P(X|C)
$$

根据朴素贝叶斯的条件独立假设，可以将文本中各个单词的条件概率进行组合。

对于词频特征，多项式朴素贝叶斯不仅考虑某个单词是否出现，还考虑该单词在文本中出现的次数。

本实验主要完成以下步骤：

1. 计算类别的先验概率；
2. 统计不同类别中的单词出现次数；
3. 计算单词的条件概率；
4. 使用拉普拉斯平滑解决零概率问题；
5. 根据后验概率完成文本分类。

### 4.1 计算类别先验概率

类别先验概率表示在还没有观察短信具体内容之前，一条短信属于某个类别的概率。

可以直接根据训练集中不同类别的样本数量进行估计：

$$
P(C)=\frac{\text{类别 C 的样本数量}}{\text{训练集总样本数量}}
$$

In [20]:
class_counts = y_train.value_counts()

prior_ham = class_counts["ham"] / len(y_train)
prior_spam = class_counts["spam"] / len(y_train)

print(f"P(ham)  = {prior_ham:.4f}")
print(f"P(spam) = {prior_spam:.4f}")

P(ham)  = 0.8658
P(spam) = 0.1342


In [10]:
y_train_array = y_train.to_numpy()

ham_mask = y_train_array == "ham"
spam_mask = y_train_array == "spam"

print("ham 训练样本数：", ham_mask.sum())
print("spam 训练样本数：", spam_mask.sum())

ham 训练样本数： 3859
spam 训练样本数： 598


In [11]:
ham_word_counts = np.asarray(
    X_train_counts[ham_mask].sum(axis=0)
).flatten()

spam_word_counts = np.asarray(
    X_train_counts[spam_mask].sum(axis=0)
).flatten()

print("ham_word_counts 形状：", ham_word_counts.shape)
print("spam_word_counts 形状：", spam_word_counts.shape)

print("\nham 中所有单词出现总次数：", ham_word_counts.sum())
print("spam 中所有单词出现总次数：", spam_word_counts.sum())

ham_word_counts 形状： (7668,)
spam_word_counts 形状： (7668,)

ham 中所有单词出现总次数： 49808
spam 中所有单词出现总次数： 14021


In [12]:
feature_names = vectorizer.get_feature_names_out()

top_spam_indices = np.argsort(spam_word_counts)[-20:][::-1]

print("spam 类别中出现次数最多的 20 个词：")

for index in top_spam_indices:
    print(
        f"{feature_names[index]:15s} "
        f"{spam_word_counts[index]}"
    )

spam 类别中出现次数最多的 20 个词：
to              561
call            281
you             243
your            215
free            180
for             166
the             165
now             155
or              148
is              136
txt             133
ur              118
have            110
from            109
on              108
stop            98
and             97
text            96
claim           94
mobile          89


In [13]:
top_ham_indices = np.argsort(ham_word_counts)[-20:][::-1]

print("ham 类别中出现次数最多的 20 个词：")

for index in top_ham_indices:
    print(
        f"{feature_names[index]:15s} "
        f"{ham_word_counts[index]}"
    )

ham 类别中出现次数最多的 20 个词：
you             1517
to              1226
the             872
in              663
and             656
my              614
me              594
is              577
it              559
that            439
of              402
for             400
so              350
have            345
but             341
can             340
are             335
your            335
on              324
not             314


### 4.2 拉普拉斯平滑与条件概率

如果某个单词在某一类别的训练数据中从未出现，通过极大似然估计得到的条件概率将为 0。

由于朴素贝叶斯需要组合多个单词的条件概率，一个条件概率为 0 可能导致整个概率乘积变为 0，这就是零概率问题。

为解决该问题，本实验使用拉普拉斯平滑：

$$
P(w_i|C)
=
\frac{N_{i,C}+\alpha}
{N_C+\alpha V}
$$

其中：

- $N_{i,C}$：单词 $w_i$ 在类别 $C$ 中出现的次数；
- $N_C$：类别 $C$ 中所有单词出现的总次数；
- $V$：词汇表大小；
- $\alpha$：平滑参数。

当 $\alpha=1$ 时，即为拉普拉斯平滑。

In [14]:
vocab_size = len(feature_names)

print("词汇表大小 V：", vocab_size)

词汇表大小 V： 7668


In [15]:
alpha = 1

ham_total_words = ham_word_counts.sum()
spam_total_words = spam_word_counts.sum()

ham_word_prob = (
    ham_word_counts + alpha
) / (
    ham_total_words + alpha * vocab_size
)

spam_word_prob = (
    spam_word_counts + alpha
) / (
    spam_total_words + alpha * vocab_size
)

print("ham 条件概率数量：", len(ham_word_prob))
print("spam 条件概率数量：", len(spam_word_prob))

print("\nham 条件概率之和：", ham_word_prob.sum())
print("spam 条件概率之和：", spam_word_prob.sum())

ham 条件概率数量： 7668
spam 条件概率数量： 7668

ham 条件概率之和： 1.0
spam 条件概率之和： 0.9999999999999998


### 4.3 对数概率

由于文本中包含大量单词，而每个单词的条件概率通常很小，直接将大量小概率相乘容易产生数值下溢。

因此实际计算时使用对数概率，将概率的乘法转换为加法。

由于对数函数是单调递增函数，因此取对数不会改变不同类别概率之间的大小关系，也不会改变最终分类结果。

In [22]:
log_prior_ham = np.log(prior_ham)
log_prior_spam = np.log(prior_spam)

log_ham_word_prob = np.log(ham_word_prob)
log_spam_word_prob = np.log(spam_word_prob)

In [23]:
#预测函数
def predict_naive_bayes(X):
    predictions = []

    for x in X:
        ham_score = (
            log_prior_ham
            + x.dot(log_ham_word_prob)[0]
        )

        spam_score = (
            log_prior_spam
            + x.dot(log_spam_word_prob)[0]
        )

        if spam_score > ham_score:
            predictions.append("spam")
        else:
            predictions.append("ham")

    return np.array(predictions)

In [24]:
new_message = [
    "Congratulations! You have won a free prize. Call now!"
]

new_message_counts = vectorizer.transform(new_message)

prediction = predict_naive_bayes(new_message_counts)

print("短信内容：", new_message[0])
print("预测类别：", prediction[0])

短信内容： Congratulations! You have won a free prize. Call now!
预测类别： spam


In [25]:
new_message = [
    "Are we still meeting for dinner tonight?"
]

new_message_counts = vectorizer.transform(new_message)

prediction = predict_naive_bayes(new_message_counts)

print("短信内容：", new_message[0])
print("预测类别：", prediction[0])

短信内容： Are we still meeting for dinner tonight?
预测类别： ham


In [26]:
y_pred_manual = predict_naive_bayes(
    X_test_counts
)

manual_accuracy = accuracy_score(
    y_test,
    y_pred_manual
)

print(
    f"手写朴素贝叶斯准确率："
    f"{manual_accuracy:.2%}"
)

手写朴素贝叶斯准确率：98.74%


## 5. 零概率问题与拉普拉斯平滑实验

为了进一步理解拉普拉斯平滑的作用，本实验从训练集词汇表中寻找在某一类别中从未出现、但在另一个类别中出现过的单词。

如果不进行平滑，根据极大似然估计，该单词在对应类别下的条件概率将为 0。在朴素贝叶斯的概率连乘过程中，一个概率为 0 就可能导致整个概率乘积变为 0。

下面通过真实数据观察这一现象。

In [27]:
zero_in_ham = np.where(
    (ham_word_counts == 0) &
    (spam_word_counts > 0)
)[0]

zero_in_spam = np.where(
    (spam_word_counts == 0) &
    (ham_word_counts > 0)
)[0]

print(
    "只在 spam 中出现的词数量：",
    len(zero_in_ham)
)

print(
    "只在 ham 中出现的词数量：",
    len(zero_in_spam)
)

只在 spam 中出现的词数量： 1621
只在 ham 中出现的词数量： 5126


In [28]:
print("部分只在 spam 中出现的词：")

for index in zero_in_ham[:20]:
    print(
        feature_names[index],
        "ham:",
        ham_word_counts[index],
        "spam:",
        spam_word_counts[index]
    )

部分只在 spam 中出现的词：
00 ham: 0 spam: 10
000 ham: 0 spam: 21
008704050406 ham: 0 spam: 1
0089 ham: 0 spam: 1
0121 ham: 0 spam: 1
01223585236 ham: 0 spam: 1
01223585334 ham: 0 spam: 2
02 ham: 0 spam: 6
0207 ham: 0 spam: 3
02072069400 ham: 0 spam: 1
02073162414 ham: 0 spam: 2
02085076972 ham: 0 spam: 1
021 ham: 0 spam: 1
03 ham: 0 spam: 10
04 ham: 0 spam: 9
0430 ham: 0 spam: 1
05 ham: 0 spam: 5
050703 ham: 0 spam: 2
0578 ham: 0 spam: 2
06 ham: 0 spam: 7


In [29]:
print("部分只在 ham 中出现的词：")

for index in zero_in_spam[:20]:
    print(
        feature_names[index],
        "ham:",
        ham_word_counts[index],
        "spam:",
        spam_word_counts[index]
    )

部分只在 ham 中出现的词：
000pes ham: 1 spam: 0
0quit ham: 1 spam: 0
1030 ham: 2 spam: 0
1120 ham: 1 spam: 0
116 ham: 1 spam: 0
1205 ham: 1 spam: 0
128 ham: 1 spam: 0
130 ham: 1 spam: 0
140 ham: 1 spam: 0
1405 ham: 1 spam: 0
15 ham: 3 spam: 0
15pm ham: 1 spam: 0
1680 ham: 1 spam: 0
180 ham: 1 spam: 0
1843 ham: 1 spam: 0
1apple ham: 1 spam: 0
1cup ham: 1 spam: 0
1hr ham: 2 spam: 0
1im ham: 1 spam: 0
1lemon ham: 1 spam: 0


In [30]:
index = zero_in_ham[0]

word = feature_names[index]

print("选择的单词：", word)
print("ham 中出现次数：", ham_word_counts[index])
print("spam 中出现次数：", spam_word_counts[index])

选择的单词： 00
ham 中出现次数： 0
spam 中出现次数： 10


In [31]:
prob_without_smoothing = (
    ham_word_counts[index] / ham_total_words
)

print(
    "不使用平滑：",
    f"P({word}|ham) =",
    prob_without_smoothing
)

不使用平滑： P(00|ham) = 0.0


In [32]:
prob_with_smoothing = (
    ham_word_counts[index] + 1
) / (
    ham_total_words + vocab_size
)

print(
    "使用拉普拉斯平滑：",
    f"P({word}|ham) =",
    prob_with_smoothing
)

使用拉普拉斯平滑： P(00|ham) = 1.739856635813209e-05


In [33]:
print("单词：", word)

print(
    "平滑前：",
    prob_without_smoothing
)

print(
    "平滑后：",
    prob_with_smoothing
)

单词： 00
平滑前： 0.0
平滑后： 1.739856635813209e-05


### 5.1 实验现象

在训练数据中，部分单词只在某一个类别中出现，而在另一个类别中的出现次数为 0。

如果直接使用极大似然估计：

$$
P(w|C)=\frac{N_{w,C}}{N_C}
$$

当 $N_{w,C}=0$ 时：

$$
P(w|C)=0
$$

由于朴素贝叶斯需要组合多个单词的条件概率，一个条件概率为 0 就可能使整个概率乘积变为 0。

加入拉普拉斯平滑后：

$$
P(w|C)=
\frac{N_{w,C}+1}
{N_C+V}
$$

即使某个单词在该类别中从未出现，其条件概率也不再为 0。

因此，拉普拉斯平滑能够有效解决训练数据中未出现特征导致的零概率问题。

## 6. 手写朴素贝叶斯分类结果

为了评价手写多项式朴素贝叶斯分类器的性能，本实验使用测试集进行预测，并通过 Accuracy、混淆矩阵以及 Precision、Recall 和 F1-score 等指标进行评价。

In [34]:
print(
    f"测试集 Accuracy："
    f"{manual_accuracy:.2%}"
)

测试集 Accuracy：98.74%


In [35]:
cm = confusion_matrix(
    y_test,
    y_pred_manual,
    labels=["ham", "spam"]
)

cm_df = pd.DataFrame(
    cm,
    index=["真实 ham", "真实 spam"],
    columns=["预测 ham", "预测 spam"]
)

cm_df

,预测 ham,预测 spam
真实 ham,964,2
真实 spam,12,137


In [36]:
print(
    classification_report(
        y_test,
        y_pred_manual,
        digits=4
    )
)

              precision    recall  f1-score   support

         ham     0.9877    0.9979    0.9928       966
        spam     0.9856    0.9195    0.9514       149

    accuracy                         0.9874      1115
   macro avg     0.9867    0.9587    0.9721      1115
weighted avg     0.9874    0.9874    0.9873      1115



## 7. sklearn MultinomialNB 对比实验

为了验证手写多项式朴素贝叶斯分类器的实现效果，本实验进一步使用 scikit-learn 提供的 `MultinomialNB` 进行对比。

sklearn 模型使用相同的训练集、测试集、词频特征以及拉普拉斯平滑参数 $\alpha=1$。

In [37]:
sklearn_nb = MultinomialNB(
    alpha=1.0
)

sklearn_nb.fit(
    X_train_counts,
    y_train
)

y_pred_sklearn = sklearn_nb.predict(
    X_test_counts
)

sklearn_accuracy = accuracy_score(
    y_test,
    y_pred_sklearn
)

print(
    f"手写朴素贝叶斯准确率："
    f"{manual_accuracy:.2%}"
)

print(
    f"sklearn MultinomialNB 准确率："
    f"{sklearn_accuracy:.2%}"
)

print(
    "两种方法预测结果是否完全一致：",
    np.array_equal(
        y_pred_manual,
        y_pred_sklearn
    )
)

手写朴素贝叶斯准确率：98.74%
sklearn MultinomialNB 准确率：98.74%
两种方法预测结果是否完全一致： True


## 8. 实验结果分析

### 8.1 数据集与类别分布分析

本实验使用 SMS Spam Collection 数据集，共包含 5572 条短信，其中正常短信（ham）4825 条，垃圾短信（spam）747 条。

训练集包含 4457 条短信，其中：

- ham：3859 条；
- spam：598 条。

对应的类别先验概率为：

$$
P(ham)=0.8658
$$

$$
P(spam)=0.1342
$$

可以看出，该数据集存在较明显的类别不平衡，正常短信数量明显多于垃圾短信。因此，在评价模型时不能只关注 Accuracy，还需要结合 Precision、Recall、F1-score 和混淆矩阵进行分析。

### 8.2 文本词频特征分析

使用 CountVectorizer 将原始短信转换为词频向量后，训练集共得到 7668 个不同的词特征。

在 spam 类别中出现次数较多的词包括：

`to`、`call`、`you`、`your`、`free`、`now`、`txt`、`stop`、`claim`、`mobile` 等。

其中 `free`、`call`、`txt`、`claim` 和 `mobile` 等词具有较明显的垃圾短信特征。

在 ham 类别中出现次数较多的词包括：

`you`、`to`、`the`、`my`、`me`、`it`、`that`、`so`、`have`、`can` 等。

这些词更多地出现在日常交流中。

同时可以发现，`to`、`you`、`the` 等高频词可能同时大量出现在两个类别中。因此，一个词出现次数多并不一定意味着它具有较强的类别区分能力，分类时还需要考虑该词在不同类别中的条件概率。

### 8.3 零概率问题与拉普拉斯平滑

实验发现，训练集词汇表中有 1621 个词只在 spam 类别中出现，有 5126 个词只在 ham 类别中出现。

例如，单词 `00` 在训练集中的出现次数为：

- ham：0 次；
- spam：10 次。

如果直接采用极大似然估计而不进行平滑，则：

$$
P(00|ham)=0
$$

由于朴素贝叶斯需要组合多个特征的条件概率，一个条件概率为 0 就可能导致整个概率乘积变为 0，从而产生零概率问题。

加入拉普拉斯平滑后：

$$
P(00|ham)=1.7398566\times10^{-5}
$$

条件概率由 0 变为一个很小但非零的概率。

实验结果直观验证了拉普拉斯平滑能够有效避免训练集中未出现的“类别-特征”组合导致条件概率为 0 的问题，提高模型对未充分观察到的特征组合的处理能力。

### 8.4 手写朴素贝叶斯分类结果

手写多项式朴素贝叶斯模型在 1115 条测试样本上的分类准确率为：

$$
Accuracy=98.74\%
$$

混淆矩阵为：

| 真实类别 | 预测 ham | 预测 spam |
| --- | ---: | ---: |
| ham | 964 | 2 |
| spam | 12 | 137 |

在 966 条真实 ham 短信中，有 964 条被正确分类，仅有 2 条被错误分类为 spam。

在 149 条真实 spam 短信中，有 137 条被正确分类，12 条被错误分类为 ham。

模型对 ham 类别的 Recall 达到 0.9979，说明绝大多数正常短信都能够被正确识别。

spam 类别的 Precision 为 0.9856，Recall 为 0.9195，F1-score 为 0.9514。相比 ham 类别，模型对 spam 的召回率相对较低，说明仍有少量垃圾短信被错误识别为正常短信。

由于数据集中 ham 样本数量明显多于 spam，因此仅观察 98.74% 的总体准确率并不足以全面评价模型。结合混淆矩阵和分类报告可以看出，该模型不仅总体准确率较高，对少数类别 spam 也取得了较好的分类效果。

### 8.5 手写模型与 sklearn 对比

在相同训练集、测试集、词频特征以及拉普拉斯平滑参数 $\alpha=1$ 的条件下，实验结果如下：

| 模型 | Accuracy |
| --- | ---: |
| 手写 Multinomial Naive Bayes | 98.74% |
| sklearn MultinomialNB | 98.74% |

进一步比较两个模型在测试集上的预测结果，得到：

`True`

说明在当前实验设置下，手写朴素贝叶斯与 sklearn MultinomialNB 的预测结果完全一致。

该结果验证了本实验从零实现的多项式朴素贝叶斯分类过程，包括类别先验概率计算、词频统计、拉普拉斯平滑、对数概率计算以及最终分类决策等步骤。

## 9. 实验总结

本实验基于 SMS Spam Collection 数据集完成了垃圾短信分类任务，并从零实现了多项式朴素贝叶斯分类算法。

首先，使用 CountVectorizer 将原始短信文本转换为词频向量，使文本数据能够用于机器学习模型。通过这一过程进一步理解了词袋模型以及文本特征数值化的基本方法。

随后，根据训练数据分别统计 ham 和 spam 类别的样本数量和单词出现次数，计算类别先验概率以及单词在不同类别下的条件概率，并根据朴素贝叶斯的条件独立假设完成分类。

实验还重点研究了零概率问题和拉普拉斯平滑。实际数据中存在大量只在某一个类别中出现的单词，例如单词 `00` 在 ham 类别中的出现次数为 0。如果直接使用极大似然估计，其条件概率将为 0；加入拉普拉斯平滑后，该概率变为非零值，从而避免零概率对分类结果造成影响。

为了避免大量小概率相乘导致数值下溢，模型进一步采用对数概率进行计算，将概率乘法转换为对数概率的加法。

最终，手写多项式朴素贝叶斯在测试集上取得了 98.74% 的准确率。其中 ham 类别的 F1-score 为 0.9928，spam 类别的 F1-score 为 0.9514。模型仅将 2 条正常短信误判为垃圾短信，同时有 12 条垃圾短信被误判为正常短信。

在相同实验条件下，sklearn MultinomialNB 同样取得了 98.74% 的准确率，并且与手写模型的预测结果完全一致，验证了手写实现的正确性。

通过本次实验，加深了对贝叶斯分类、条件独立假设、先验概率、条件概率、极大似然估计、零概率问题、拉普拉斯平滑以及对数概率计算等概念的理解，同时掌握了朴素贝叶斯在文本分类任务中的基本应用流程。